In [ ]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client.http.models import VectorParams

from langchain.AgentPractice import result

### Ingestion Process

In [ ]:
# Load Document — resolve `data` whether cwd is repo root or rag_practice/
_cwd = Path.cwd().resolve()
print(_cwd)
if (_cwd / "data").is_dir():
    ROOT_PATH = _cwd / "data"
elif (_cwd.parent / "data").is_dir():
    ROOT_PATH = _cwd.parent / "data"
else:
    ROOT_PATH = _cwd / "data"
doc_list = ["kl_tourism.pdf","Chennai.1", "hyd_data.pdf", "pu_tourism.txt"]


def parse_document(docs: list[str]) -> list[Document]:
    all_docs:list[Document] = []
    for doc in docs:
        file_path = ROOT_PATH / doc
        if not file_path.is_file():
            print(f"File Path Issue: {file_path}, continuing with the rest of file")
            continue
        _, ext = os.path.splitext(doc.lower())
        if ext == ".txt":
            all_docs.extend(extract_content(TextLoader(file_path)))
        elif ext == ".pdf":
            all_docs.extend(extract_content(PyPDFLoader(file_path)))
    return all_docs


def extract_content(loader: BaseLoader) -> list[Document]:
    docs:list[Document]= []
    try:
        docs = loader.load()
        print(f"Content Type: {type(loader).__name__} / Content Length: {len(docs)}")
    except Exception as e:
        print(f"Extraction Error! {e}")
    return docs

merged_docs = parse_document(doc_list)
if not merged_docs:
    print("No Docs available to ingest")
else:
    print(f"Total Docs size: {len(merged_docs)}")

In [ ]:
#Text Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunked_docs:list[Document] = splitter.split_documents(merged_docs)
print(len(chunked_docs))

In [ ]:
#Define Embeddings & Collection name
doc_embedding_model = OllamaEmbeddings(model="embeddinggemma:300m",base_url="http://127.0.0.1:11434",)
db_collection_name = "travel_docs"

### Chroma DB

In [ ]:
# Create vector DB and store docs
from langchain_chroma.vectorstores import Chroma

store_directory = (ROOT_PATH / "db").resolve()
os.makedirs(store_directory, exist_ok=True)

persist_path = store_directory.as_posix()

#Initialize Chroma DB
chroma_vector_db = Chroma(collection_name=db_collection_name, embedding_function=doc_embedding_model, persist_directory=persist_path)



In [ ]:
## Quick for prototype
# chroma_vector_db = Chroma.from_documents(
#     documents=chunked_docs,
#     embedding=doc_embedding_model,
#     collection_name="travel_docs",
#     persist_directory=persist_path
#)

In [ ]:
#add docs to the chroma db
ids = chroma_vector_db.add_documents(documents=chunked_docs)
print(len(ids))
chroma_vector_db.get()["ids"][1]
#chroma_vector_db.get("4fa5d9e0-9758-4730-b712-5061c312ffe9")

In [ ]:
chroma_vector_db.similarity_search("tell me something about pondicherry beach?",k=2)

### Qdrant DB

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant.vectorstores import QdrantClient
from qdrant_client.http.models import VectorParams, Distance

# Qdrant Client & Create Collection
qdrant_client = QdrantClient(host="localhost")
if not qdrant_client.collection_exists(collection_name=db_collection_name):
    qdrant_client.create_collection(collection_name=db_collection_name, vectors_config=VectorParams(size=768, distance=Distance.COSINE))

# Create Qdrant Vector Store with the collection
qd_vector_store = QdrantVectorStore(client=qdrant_client, collection_name=db_collection_name, embedding=doc_embedding_model)

## Quick Implementations without explicit client
# QdrantVectorStore.from_documents(
#         documents=chunked_docs,
#         embedding=doc_embedding_model,
#         collection_name=db_collection_name,
#         url="http://localhost:6333"
#     )

In [ ]:
# Add Data -- Verify the Duplicate logics -- TODO:
qd_vector_store.add_documents(documents=chunked_docs)

In [ ]:
result = qd_vector_store.similarity_search("tell about aurovile?", k=3)
print(len(result))
result